In [1]:
import sqlite3
import pandas as pd

DB_PATH = '../atp/data/atp.db'

In [2]:
conn = sqlite3.connect(DB_PATH)

df = pd.read_sql_query("""
    SELECT p1_score, p2_score, winner_id, p1_id, number_of_sets, surface
    FROM matches
    WHERE match_status = 'F'
      AND reason IS NULL
      AND is_qualifier = 0
      AND p1_score IS NOT NULL AND p2_score IS NOT NULL
      AND number_of_sets >= 2
""", conn)

conn.close()
print(f"Finished matches loaded: {len(df)}")

Finished matches loaded: 7498


In [3]:
def parse_set1(score):
    try:
        return int(str(score).split(';')[0])
    except (ValueError, IndexError):
        return None

df['s1_p1'] = df['p1_score'].apply(parse_set1)
df['s1_p2'] = df['p2_score'].apply(parse_set1)

df = df.dropna(subset=['s1_p1', 's1_p2'])
df = df[df['s1_p1'] != df['s1_p2']]

df['set1_winner_is_p1'] = df['s1_p1'] > df['s1_p2']
df['match_winner_is_p1'] = df['winner_id'] == df['p1_id']
df['set1_winner_won_match'] = df['set1_winner_is_p1'] == df['match_winner_is_p1']

print(f"Analysable matches: {len(df)}")

Analysable matches: 7497


In [4]:
# Overall
total = len(df)
won = df['set1_winner_won_match'].sum()
print(f"Overall: {won} / {total}  ({100*won/total:.1f}%)")
print()

# By sets played (2 = straight sets, 3/4/5 = went the distance)
print("By sets played:")
for n, g in df.groupby('number_of_sets'):
    w = g['set1_winner_won_match'].sum()
    label = "straight sets" if n == 2 else f"went to {n} sets"
    print(f"  {n} sets played ({label}): {w} / {len(g)}  ({100*w/len(g):.1f}%)")
print()

# By surface
print("By surface:")
for s, g in df.groupby('surface'):
    w = g['set1_winner_won_match'].sum()
    print(f"  {s}: {w} / {len(g)}  ({100*w/len(g):.1f}%)")

Overall: 5936 / 7497  (79.2%)

By sets played:
  2 sets played (straight sets): 3389 / 3389  (100.0%)
  3 sets played (went to 3 sets): 2134 / 3372  (63.3%)
  4 sets played (went to 4 sets): 205 / 333  (61.6%)
  5 sets played (went to 5 sets): 208 / 403  (51.6%)

By surface:
  Clay: 1909 / 2441  (78.2%)
  Grass: 639 / 808  (79.1%)
  Hard: 3388 / 4248  (79.8%)
